# 20260106_E_2_1_FCMC — side1_cam SLEAP 推理（3-node bottom-up 模型）

**目的**：用 `retrain_20260810_3node_reviewed` 的 side bottom-up 模型
（`s14.bottomup_3node.reviewed_side_f24_ms64_paf8`，骨架 mouth–pectoral–tail）
对 `20260106_E_2_1_FCMC` 的 side1_cam 全长视频做推理，输出 `side1_cam.slp`。
这是 [20260106_E_2_1_FCMC_3d_reconstruction_workflow.md](20260106_E_2_1_FCMC_3d_reconstruction_workflow.md) 的 Stage 1（单视角）。

**路径设计（换地址 / 换电脑时）**：
- 仓库根目录由步骤 2 自动向上查找（找 `pyproject.toml`），仓库克隆到哪里都能用，不写死 `C:\Users\...`。
- 数据所在的网络盘路径只写在**一个文件**里：优先 `configs/paths.<用户名>.toml`，没有就用 `configs/paths.local.toml`。
  换电脑时只改那个 toml（从 `configs/paths.example.toml` 复制），notebook 本身不用改。
- 模型、视频、输出都用 toml 里的根目录 + 下面的相对路径拼出来。
- 新地址需要重新建虚拟环境（`uv sync --python 3.11`），并在 Jupyter 里选该仓库 `.venv` 的内核。

**测速结论（2026-10-26，RTX 5070 Ti 16 GB，side1_cam，帧 172400–173599）**：

| batch size | 速度 | 显存峰值 |
|---|---|---|
| 2 | 31.4 FPS | ~4 GB |
| 4 | 33.0 FPS | — |
| 6 | 33.1 FPS | — |
| 8 | 33.9 FPS | 13.9 GB |
| 12 | **1.35 FPS**（显存溢出到内存，慢 25 倍） | 15.9 GB |

batch ≥ 4 速度基本一样（瓶颈在视频解码和 bottom-up 的 PAF 分组，不在 GPU），
batch 8 离显存上限太近，鱼多/重叠的帧可能溢出。**用 batch 4**。
全长 348600 帧 ÷ ~33 FPS ≈ **2.9 小时**。

## 步骤 1 — 参数

只需要改这一格。`MODEL_SUBPATH` 是相对 toml 里 `sleap_models_dir` 的路径；
`RUN_INFERENCE = True` 时步骤 5 才会真正启动全长推理（防止“全部运行”时误启动 3 小时的任务）。

In [1]:
EXPERIMENT    = "20260106_E_2_1_FCMC"
VIEW          = "side1_cam"   # top_cam / side1_cam / side2_cam
MODEL_SUBPATH = "retrain_20260810_3node_reviewed/side/s14.bottomup_3node.reviewed_side_f24_ms64_paf8"
ROUND_TAG     = "s14_bottomup_3node_reviewed"   # 输出子目录名，后续 3D 步骤的 --sleap-subdir

BATCH_SIZE    = 4      # 见上面测速表；不要用 12
MAX_INSTANCES = 6      # 故意多于 3 条鱼，3D 阶段 --num-candidates 6 依赖它
EXPECTED_FPS  = 33.0   # 本机实测，用于估算时间

PATHS_CONFIG  = None   # None = 自动选择；也可以写某个 toml 的路径
RUN_INFERENCE = False  # 确认后改成 True 再运行步骤 5

## 步骤 2 — 定位仓库根目录和本机路径配置

- 输入：当前工作目录（notebook 所在位置）、`configs/paths.<用户名>.toml` 或 `configs/paths.local.toml`
- 输出：`REPO`、`PATHS`（各数据根目录），并检查当前内核是否是本仓库的 `.venv`

In [2]:
import getpass
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "scripts" / "common" / "local_paths.py").exists():
            return d
    raise FileNotFoundError(f"找不到仓库根目录（从 {start} 向上没有 pyproject.toml）")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.common.local_paths import load_local_paths

if PATHS_CONFIG is not None:
    cfg_path = Path(PATHS_CONFIG)
else:
    user_cfg = REPO / "configs" / f"paths.{getpass.getuser()}.toml"
    cfg_path = user_cfg if user_cfg.exists() else REPO / "configs" / "paths.local.toml"
PATHS = load_local_paths(cfg_path)
if PATHS is None:
    raise FileNotFoundError(f"路径配置不存在：{cfg_path}（从 configs/paths.example.toml 复制一份再改）")

print("仓库根目录 :", REPO)
print("路径配置   :", cfg_path)
print("Python     :", sys.executable)
if not Path(sys.executable).resolve().is_relative_to((REPO / ".venv").resolve()):
    print("⚠️ 当前内核不是本仓库的 .venv —— 请在 Jupyter 里切换内核")

仓库根目录 : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline
路径配置   : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\configs\paths.ReiterU.toml
Python     : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\.venv\Scripts\python.exe


## 步骤 3 — 解析输入/输出路径并检查

- 输入：`medaka_data_dir/<EXPERIMENT>/<VIEW>/cam*.mp4`、`sleap_models_dir/<MODEL_SUBPATH>`、`experiment_metadata.json`
- 输出：`VIDEO`、`MODEL_DIR`、`OUT_SLP = sleap_output_dir/<EXPERIMENT>/<ROUND_TAG>/<VIEW>.slp`，以及时间估算
- 任何一项不存在会直接报错，并提示该改 toml 里的哪个键

In [3]:
import json


def need(path: Path, key: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"不存在：{path}\n→ 检查 {cfg_path.name} 里的 {key}")
    return path


exp_dir = need(PATHS.medaka_data_dir / EXPERIMENT, "medaka_data_dir")
videos = sorted((exp_dir / VIEW).glob("cam*.mp4"))
if len(videos) != 1:
    raise FileNotFoundError(f"{exp_dir / VIEW} 里应正好有 1 个 cam*.mp4，找到 {len(videos)} 个")
VIDEO = videos[0]

MODEL_DIR = need(PATHS.sleap_models_dir / MODEL_SUBPATH, "sleap_models_dir")
for f in ("training_config.yaml", "best.ckpt"):
    need(MODEL_DIR / f, "sleap_models_dir / MODEL_SUBPATH")

OUT_DIR = need(PATHS.sleap_output_dir, "sleap_output_dir") / EXPERIMENT / ROUND_TAG
OUT_SLP = OUT_DIR / f"{VIEW}.slp"
LOG_PATH = OUT_DIR / f"{VIEW}.inference.log"
PID_PATH = OUT_DIR / f"{VIEW}.inference.pid.json"

meta = json.loads((exp_dir / "experiment_metadata.json").read_text(encoding="utf-8"))["views"][VIEW]
n_frames = meta["total_frames"]

print("视频   :", VIDEO, f"({VIDEO.stat().st_size / 1e9:.1f} GB)")
print("模型   :", MODEL_DIR)
print("输出   :", OUT_SLP, "（已存在）" if OUT_SLP.exists() else "")
print(f"帧数   : {n_frames}（{meta['frame_rate']:.0f} fps 录制，{n_frames / meta['frame_rate'] / 60:.1f} 分钟）")
print(f"预计   : {n_frames / EXPECTED_FPS / 3600:.1f} 小时（按 {EXPECTED_FPS} FPS 推理速度）")

视频   : \\bucket.oist.jp\bucket\StephensU\medaka\medaka_202512\20260106_E_2_1_FCMC\side1_cam\cam1_side.mp4 (5.0 GB)
模型   : \\bucket.oist.jp\bucket\StephensU\medaka\sleap_models\retrain_20260810_3node_reviewed\side\s14.bottomup_3node.reviewed_side_f24_ms64_paf8
输出   : \\bucket.oist.jp\bucket\StephensU\medaka\sleap_output\20260106_E_2_1_FCMC\s14_bottomup_3node_reviewed\side1_cam.slp 
帧数   : 348600（166 fps 录制，35.0 分钟）
预计   : 2.9 小时（按 33.0 FPS 推理速度）


## 步骤 4 — 推理命令

直接调用 `sleap-nn track`（bottom-up 模型不能用 `run_round_inference.py`，那个只支持 top-down）。
不加 `--tracking`：身份追踪由后面的 `track_simple_nn` / 3D 步骤做。

In [4]:
CMD = [
    sys.executable, "-m", "sleap_nn.cli", "track",
    "--data_path", str(VIDEO),
    "--model_paths", str(MODEL_DIR),
    "--output_path", str(OUT_SLP),
    "--device", "cuda",
    "--batch_size", str(BATCH_SIZE),
    "--max_instances", str(MAX_INSTANCES),
]
print(" ".join(CMD))

C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\.venv\Scripts\python.exe -m sleap_nn.cli track --data_path \\bucket.oist.jp\bucket\StephensU\medaka\medaka_202512\20260106_E_2_1_FCMC\side1_cam\cam1_side.mp4 --model_paths \\bucket.oist.jp\bucket\StephensU\medaka\sleap_models\retrain_20260810_3node_reviewed\side\s14.bottomup_3node.reviewed_side_f24_ms64_paf8 --output_path \\bucket.oist.jp\bucket\StephensU\medaka\sleap_output\20260106_E_2_1_FCMC\s14_bottomup_3node_reviewed\side1_cam.slp --device cuda --batch_size 4 --max_instances 6


## 步骤 5 — 启动全长推理（后台进程）

- 以独立后台进程运行，关掉 notebook / 内核也不会中断；日志写到 `<VIEW>.inference.log`，进程号写到 `<VIEW>.inference.pid.json`
- 输出已存在或任务已在跑时会跳过
- 任务管理器里会看到两个 python 进程（venv 启动器 + 真正的解释器），是正常现象，不要杀

In [5]:
import subprocess
import time

import psutil


def running_pid():
    if not PID_PATH.exists():
        return None
    pid = json.loads(PID_PATH.read_text())["pid"]
    return pid if psutil.pid_exists(pid) else None


if OUT_SLP.exists():
    print("跳过：输出已存在", OUT_SLP)
elif running_pid():
    print("跳过：已经在运行，PID", running_pid())
elif not RUN_INFERENCE:
    print("未启动：确认后把步骤 1 的 RUN_INFERENCE 改成 True，再运行本格")
else:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    log = open(LOG_PATH, "w", encoding="utf-8")
    log.write(f"# cmd: {' '.join(CMD)}\n# started: {time.ctime()}\n\n")
    log.flush()
    flags = subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS if os.name == "nt" else 0
    proc = subprocess.Popen(CMD, stdout=log, stderr=subprocess.STDOUT, cwd=REPO,
                            creationflags=flags, start_new_session=os.name != "nt")
    PID_PATH.write_text(json.dumps({"pid": proc.pid, "started": time.time(), "cmd": CMD}, indent=2))
    print(f"已启动 PID {proc.pid}，开始于 {time.ctime()}，预计 {n_frames / EXPECTED_FPS / 3600:.1f} 小时后完成")

未启动：确认后把步骤 1 的 RUN_INFERENCE 改成 True，再运行本格


## 步骤 6 — 查看进度

随时重新运行本格。sleap-nn 会缓冲输出，log 在跑完前可能只有表头；
用“已运行时间 / 预计时间”和 GPU 占用判断进度，`<VIEW>.slp` 出现即完成。

In [6]:
if PID_PATH.exists():
    info = json.loads(PID_PATH.read_text())
    elapsed = time.time() - info["started"]
    total = n_frames / EXPECTED_FPS
    state = "运行中" if psutil.pid_exists(info["pid"]) else "已结束"
    print(f"PID {info['pid']}：{state}，已运行 {elapsed / 3600:.2f} h / 预计 {total / 3600:.1f} h（约 {min(elapsed / total, 1):.0%}）")
else:
    print("尚未启动")
print("输出 .slp :", "已生成" if OUT_SLP.exists() else "还没有")
print(subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
                      "--format=csv"], capture_output=True, text=True).stdout)
if LOG_PATH.exists():
    print("log 最后几行：")
    print("\n".join(LOG_PATH.read_text(encoding="utf-8", errors="replace").replace("\r", "\n").splitlines()[-5:]))

尚未启动
输出 .slp : 还没有
utilization.gpu [%], memory.used [MiB], memory.total [MiB]
0 %, 0 MiB, 16303 MiB



## 步骤 7 — 结果检查

- 输入：`OUT_SLP`
- 输出：骨架节点、预测帧数（应为全部帧数）、每帧实例数分布（大多数帧应 ≥ 3）
- 之后按 workflow.md 的 1.3 节做叠加视频目视检查

In [7]:
import collections

import sleap_io as sio

if OUT_SLP.exists():
    labels = sio.load_slp(str(OUT_SLP))
    counts = collections.Counter(len(f.instances) for f in labels)
    print("骨架     :", [n.name for n in labels.skeleton.nodes])
    print(f"预测帧数 : {len(labels)} / {n_frames}")
    print("实例数分布:", sorted(counts.items()))
else:
    print("还没有输出：", OUT_SLP)

还没有输出： \\bucket.oist.jp\bucket\StephensU\medaka\sleap_output\20260106_E_2_1_FCMC\s14_bottomup_3node_reviewed\side1_cam.slp
